In [52]:
import numpy as np
import pandas as pd
import ast
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv(
    r'C:\Users\Lenovo\PycharmProjects\Movie_Recommender_system_with_Sentiment_analysis\data\tmdb_5000_movies.csv'
)

credits = pd.read_csv(
    r'C:\Users\Lenovo\PycharmProjects\Movie_Recommender_system_with_Sentiment_analysis\data\tmdb_5000_credits.csv'
)

mymovies = pd.read_csv(
    r'C:\Users\Lenovo\PycharmProjects\Movie_Recommender_system_with_Sentiment_analysis\data\mymoviedb.csv',
    engine='python',
    encoding='latin1',
    on_bad_lines='skip'
)

imdb = pd.read_csv(
    r'C:\Users\Lenovo\PycharmProjects\Movie_Recommender_system_with_Sentiment_analysis\data\movies_metadata.csv',
    engine='python',
    encoding='latin1',
    on_bad_lines='skip'
)

In [53]:

# ===============================
# Merge TMDB datasets
# ===============================
movies = movies.merge(credits, on='title')

In [54]:

# ===============================
# Safe merge with poster dataset
# ===============================

# create temporary lowercase column
movies['title_key'] = movies['title'].str.lower().str.strip()
mymovies['title_key'] = mymovies['title'].str.lower().str.strip()

In [55]:

# merge using temporary key
movies = movies.merge(
    mymovies[['title_key','Poster_Url']],
    on='title_key',
    how='left'
)

In [56]:
movies['title_key'] = movies['title'].str.lower().str.strip()
imdb['title_key'] = imdb['title'].str.lower().str.strip()

In [57]:
movies = movies.merge(
    imdb[['title_key','imdb_id']],
    on='title_key',
    how='left'
)

In [58]:

# remove temporary column
movies.drop(columns=['title_key'], inplace=True)

In [59]:

# ===============================
# Select useful columns
# ===============================

movies = movies[['movie_id','title','overview','genres','keywords','cast','crew','Poster_Url','vote_average','imdb_id']]

# Remove missing values
movies.dropna(inplace=True)

In [60]:

# ===============================
# Convert JSON columns to lists
# ===============================

def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [61]:


# ===============================
# Extract top 3 actors
# ===============================

def convert_cast(text):
    L = []
    counter = 0

    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

movies['cast'] = movies['cast'].apply(convert_cast)

In [62]:

# ===============================
# Extract director
# ===============================

def fetch_director(text):

    L = []

    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])

    return L

In [63]:

movies['crew'] = movies['crew'].apply(fetch_director)

# ===============================
# Remove spaces from words
# ===============================

def collapse(L):

    L1 = []

    for i in L:
        L1.append(i.replace(" ",""))

    return L1

In [64]:

movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [65]:

# ===============================
# Prepare overview
# ===============================
movies['original_overview'] = movies['overview']
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [66]:

# ===============================
# Create tags
# ===============================

movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Keep required columns
new = movies[['movie_id','title','tags','original_overview','Poster_Url','overview','cast','crew','vote_average','imdb_id']]

# Convert list → string
new['tags'] = new['tags'].apply(lambda x:" ".join(x))

# Lowercase tags only
new['tags'] = new['tags'].apply(lambda x:x.lower())

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15392\737985335.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new['tags'] = new['tags'].apply(lambda x:" ".join(x))
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15392\737985335.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new['tags'] = new['tags'].apply(lambda x:x.lower())


In [67]:

# ===============================
# Vectorization
# ===============================

cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new['tags']).toarray()

In [68]:


# ===============================
# Similarity Matrix
# ===============================

similarity = cosine_similarity(vectors)


In [70]:
import pickle

pickle.dump(new.to_dict(), open('../movies.pkl', 'wb'))
pickle.dump(similarity, open('../similarity.pkl', 'wb'))

print("Model files saved successfully!")

Model files saved successfully!


In [71]:
movies.columns

Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'Poster_Url', 'vote_average', 'imdb_id', 'original_overview', 'tags'],
      dtype='object')

In [72]:
print(movies[["title","imdb_id"]].head(20))

                                         title    imdb_id
0                                       Avatar  tt0499549
1     Pirates of the Caribbean: At World's End  tt0449088
2                                      Spectre  tt2379713
3                        The Dark Knight Rises  tt1345836
4                                  John Carter  tt0401729
5                                 Spider-Man 3  tt0413300
6                                      Tangled  tt0398286
7                                      Tangled  tt0238137
8                      Avengers: Age of Ultron  tt2395427
9       Harry Potter and the Half-Blood Prince  tt0417741
10          Batman v Superman: Dawn of Justice  tt2975590
11                            Superman Returns  tt0348150
12                           Quantum of Solace  tt0830515
13  Pirates of the Caribbean: Dead Man's Chest  tt0383574
14                             The Lone Ranger  tt0048310
15                             The Lone Ranger  tt1210819
16            